# Multi-template patching

Plots the most recent run in `experimental_outputs/multi-template/patching_results.json`,
written by `patching_multi_template.py`.

Each cell of the heatmap is one (token, layer) patch: the activation from the
human prompt is substituted into the animal prompt at that position, and the
value is the resulting YES − NO logit difference. High values mean the patched
site alone was enough to flip the model toward answering that this individual
should be prioritized — i.e. that site carries the human/animal distinction the
moral readout depends on.

This is the zero-shot moral readout. For the 5-shot HUMAN/ANIMAL category
readout see `patching.ipynb`.


In [ ]:
import plotly.express as px
import json
import configparser
import numpy as np
from nnsight import LanguageModel

RESULTS_PATH = 'experimental_outputs/multi-template/patching_results.json'


In [ ]:
with open(RESULTS_PATH, 'r') as f:
    out = json.load(f)[-1]  # most recent run
false_prompt = out['false_prompt']
logit_diffs = out['logit_diffs']
n_toks = len(logit_diffs)
model_name = out['model']
pos_token, neg_token = out.get('pos_token', 'YES'), out.get('neg_token', 'NO')

# tokenizer only -- no weights are loaded, so this runs fine on a laptop
config = configparser.ConfigParser()
config.read('config.ini')
model = LanguageModel(config[model_name]['weights_directory'])

# rows are layers, columns are token positions
logit_diffs = [[logit_diffs[i][j] for i in range(0, len(logit_diffs))[::-1]] for j in range(len(logit_diffs[0]))]
probs = [[1 / (1 + np.exp(-logit)) for logit in layer] for layer in logit_diffs]

token_ids = model.tokenizer(false_prompt)['input_ids']
tokens = [model.tokenizer.decode([token_id]) + f" ({idx})" for idx, token_id in enumerate(token_ids)]
tokens = tokens[-n_toks:]

print(f"{model_name}: {len(logit_diffs)} layers x {n_toks} patched token positions")
print(f"readout: {pos_token} - {neg_token}")


In [ ]:
fig = px.imshow(
    logit_diffs,
    x=tokens,
    labels=dict(x="Token", y="Layer", color=f"{pos_token} - {neg_token}"),
    color_continuous_scale='blues',
    title=f"Multi-template patching, {model_name}",
)
fig.show()


In [ ]:
# save alongside the other multi-template figures
import os
os.makedirs('dataexplorer/plots/multi-template', exist_ok=True)
fig.write_image('dataexplorer/plots/multi-template/patching-results.png', scale=2)
